In [1]:
# =========================
# Proper (No-Leakage) Fusion for AMDNet23 (train/valid folders)
# - Train base models on base_train split
# - Train fusion (weights/stacking) on meta_train split (from train only)
# - Evaluate final performance on VALID folder only
# =========================

import random
import numpy as np
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression

from torch.amp import autocast, GradScaler


# -------------------------
# Config
# -------------------------
@dataclass
class CFG:
    DATA_ROOT: str = "/kaggle/input/fundus-image/AMDNet23 Fundus Image Dataset for  Age-Related Macular Degeneration Disease Detection/AMDNet23 Dataset"

    IMG_SIZE: int = 224
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # split train -> base_train + meta_train (for fusion learning)
    META_SPLIT: float = 0.20  # 20% of train used for meta/fusion training

    # training
    EPOCHS_HEAD: int = 5
    EPOCHS_FINE: int = 15
    LR_HEAD: float = 3e-4
    LR_FINE: float = 1e-5
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05
    PATIENCE: int = 5

    # fusion condition
    TARGET_ACC: float = 0.98

cfg = CFG()


# -------------------------
# Reproducibility
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(cfg.SEED)


# -------------------------
# Paths (FIXED)
# -------------------------
data_root = Path(cfg.DATA_ROOT)
train_dir = data_root / "train"
valid_dir = data_root / "valid"

assert train_dir.exists(), f"Train folder not found: {train_dir}"
assert valid_dir.exists(), f"Valid folder not found: {valid_dir}"


# -------------------------
# Transforms
# -------------------------
train_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_tfms = transforms.Compose([
    transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])


# -------------------------
# Datasets
# -------------------------
# base dataset for indexing / labels
train_ds_index = datasets.ImageFolder(str(train_dir))
class_names = train_ds_index.classes
num_classes = len(class_names)
assert num_classes == 4, f"Expected 4 classes, got {num_classes}: {class_names}"

# Get targets for stratified split
targets = np.array([y for _, y in train_ds_index.samples])
idx = np.arange(len(targets))

base_idx, meta_idx = train_test_split(
    idx,
    test_size=cfg.META_SPLIT,
    stratify=targets,
    random_state=cfg.SEED
)

# Actual datasets with transforms
train_ds_full = datasets.ImageFolder(str(train_dir), transform=train_tfms)
meta_ds_full  = datasets.ImageFolder(str(train_dir), transform=eval_tfms)  # meta should be eval-like
valid_ds      = datasets.ImageFolder(str(valid_dir), transform=eval_tfms)

# Subsets
base_train_ds = Subset(train_ds_full, base_idx)
meta_train_ds = Subset(meta_ds_full, meta_idx)

# sanity check: same class mapping
assert train_ds_full.class_to_idx == valid_ds.class_to_idx, "Train/Valid class mapping mismatch"
assert train_ds_full.class_to_idx == meta_ds_full.class_to_idx, "Train/Meta class mapping mismatch"

# Loaders
base_loader = DataLoader(base_train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True)
meta_loader = DataLoader(meta_train_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                         num_workers=cfg.NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, pin_memory=True)

print("Classes:", class_names)
print("base_train size:", len(base_train_ds), "| meta_train size:", len(meta_train_ds), "| valid size:", len(valid_ds))


# -------------------------
# Model list
# -------------------------
MODEL_SPECS = [
    ("tf_efficientnetv2_s", "EffNetV2-S"),
    ("densenet121", "DenseNet121"),
    ("swin_tiny_patch4_window7_224", "Swin-Tiny"),
]

def make_model(model_name: str, num_classes: int):
    return timm.create_model(model_name, pretrained=True, num_classes=num_classes)

def set_trainable_backbone(model, train_backbone: bool):
    for p in model.parameters():
        p.requires_grad = train_backbone
    for name, p in model.named_parameters():
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            p.requires_grad = True

@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    all_probs, all_y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        all_probs.append(probs)
        all_y.append(y.numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_y, axis=0)

@torch.no_grad()
def eval_with_probs(probs, y):
    pred = probs.argmax(axis=1)
    acc = accuracy_score(y, pred)
    f1m = f1_score(y, pred, average="macro")
    return acc, f1m, pred

@torch.no_grad()
def eval_model(model, loader, device):
    probs, y = predict_proba(model, loader, device)
    acc, f1m, pred = eval_with_probs(probs, y)
    return acc, f1m, probs, y


def train_one_phase(model, train_loader, eval_loader, device, epochs, lr, weight_decay,
                    label_smoothing=0.0, patience=5):

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))

    scaler = GradScaler("cuda", enabled=device.startswith("cuda"))

    best_acc = -1.0
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=device.startswith("cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)

        scheduler.step()

        eval_acc, eval_f1, _, _ = eval_model(model, eval_loader, device)
        tr_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {ep:02d}/{epochs} | loss {tr_loss:.4f} | meta_acc {eval_acc:.4f} | meta_f1 {eval_f1:.4f}")

        if eval_acc > best_acc:
            best_acc = eval_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc


# -------------------------
# Train base models (validated on META set, not VALID)
# -------------------------
trained_models = []
meta_scores = []  # (model_name, meta_acc, meta_f1)

for timm_name, nice_name in MODEL_SPECS:
    print("\n==============================")
    print("Training:", nice_name, f"({timm_name})")
    print("==============================")

    model = make_model(timm_name, num_classes=num_classes).to(cfg.DEVICE)

    # Head training
    set_trainable_backbone(model, train_backbone=False)
    model, _ = train_one_phase(
        model, base_loader, meta_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_HEAD,
        lr=cfg.LR_HEAD,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=max(2, cfg.PATIENCE // 2),
    )

    # Fine-tuning
    set_trainable_backbone(model, train_backbone=True)
    model, _ = train_one_phase(
        model, base_loader, meta_loader, cfg.DEVICE,
        epochs=cfg.EPOCHS_FINE,
        lr=cfg.LR_FINE,
        weight_decay=cfg.WEIGHT_DECAY,
        label_smoothing=cfg.LABEL_SMOOTHING,
        patience=cfg.PATIENCE,
    )

    meta_acc, meta_f1, _, _ = eval_model(model, meta_loader, cfg.DEVICE)
    print(f"[{nice_name}] META Acc: {meta_acc:.4f} | META Macro-F1: {meta_f1:.4f}")

    trained_models.append((nice_name, model))
    meta_scores.append((nice_name, meta_acc, meta_f1))


# -------------------------
# Learn fusion on META (no leakage)
# -------------------------
# Collect META probs for each model
meta_probs = {}
meta_y = None
for name, model in trained_models:
    probs, y = predict_proba(model, meta_loader, cfg.DEVICE)
    meta_probs[name] = probs
    meta_y = y

best_single = max(meta_scores, key=lambda x: x[1])
best_single_name, best_single_acc, best_single_f1 = best_single
print("\nBest single on META:", best_single)

# Candidate 1: choose single if meets target on META
chosen_strategy = None
chosen_meta = None
weights = None

if best_single_acc >= cfg.TARGET_ACC:
    chosen_strategy = f"SingleModel:{best_single_name}"
    print(f"\n✅ Condition met on META: {best_single_name} >= {cfg.TARGET_ACC:.2f}. Using single model.")
else:
    # Candidate 2: weighted soft vote using (acc+f1)/2 weights computed on META
    weights = {}
    for n, a, f in meta_scores:
        weights[n] = max(1e-6, (a + f) / 2.0)
    s = sum(weights.values())
    for k in weights:
        weights[k] /= s

    soft_vote_meta = np.zeros_like(next(iter(meta_probs.values())))
    for n in meta_probs:
        soft_vote_meta += weights[n] * meta_probs[n]

    soft_acc, soft_f1, _ = eval_with_probs(soft_vote_meta, meta_y)
    print("\nWeighted soft-vote on META:", soft_acc, soft_f1, "weights:", weights)

    # Candidate 3: stacking on META
    X_meta = np.concatenate([meta_probs[n] for n, _ in trained_models], axis=1)
    meta_clf = LogisticRegression(max_iter=2000, n_jobs=-1, multi_class="auto")
    meta_clf.fit(X_meta, meta_y)
    stack_meta_probs = meta_clf.predict_proba(X_meta)
    stack_acc, stack_f1, _ = eval_with_probs(stack_meta_probs, meta_y)
    print("Stacking on META:", stack_acc, stack_f1)

    if stack_acc > soft_acc:
        chosen_strategy = "Stacking(LogReg)"
        chosen_meta = meta_clf
    else:
        chosen_strategy = "WeightedSoftVote"

print("\nChosen strategy (learned on META):", chosen_strategy)


# -------------------------
# Final evaluation on VALID (never used before)
# -------------------------
# Collect VALID probs
valid_probs = {}
valid_y = None
for name, model in trained_models:
    probs, y = predict_proba(model, valid_loader, cfg.DEVICE)
    valid_probs[name] = probs
    valid_y = y

if chosen_strategy.startswith("SingleModel:"):
    chosen_name = chosen_strategy.split(":", 1)[1]
    final_probs = valid_probs[chosen_name]
elif chosen_strategy == "WeightedSoftVote":
    final_probs = np.zeros_like(next(iter(valid_probs.values())))
    for n in valid_probs:
        final_probs += weights[n] * valid_probs[n]
elif chosen_strategy == "Stacking(LogReg)":
    X_valid = np.concatenate([valid_probs[n] for n, _ in trained_models], axis=1)
    final_probs = chosen_meta.predict_proba(X_valid)
else:
    raise RuntimeError("Unknown strategy.")

valid_pred = final_probs.argmax(axis=1)
val_acc = accuracy_score(valid_y, valid_pred)
val_f1m = f1_score(valid_y, valid_pred, average="macro")
cm = confusion_matrix(valid_y, valid_pred)

print("\n==============================")
print("FINAL VALID RESULTS (NO LEAKAGE)")
print("==============================")
print("Strategy:", chosen_strategy)
print(f"Valid Accuracy: {val_acc:.4f}")
print(f"Valid Macro-F1: {val_f1m:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(valid_y, valid_pred, target_names=class_names))


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Classes: ['amd', 'cataract', 'diabetes', 'normal']
base_train size: 1275 | meta_train size: 319 | valid size: 400

Training: EffNetV2-S (tf_efficientnetv2_s)


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

Epoch 01/5 | loss 3.7809 | meta_acc 0.6646 | meta_f1 0.6544
Epoch 02/5 | loss 2.2424 | meta_acc 0.6489 | meta_f1 0.6461
Epoch 03/5 | loss 1.8437 | meta_acc 0.6803 | meta_f1 0.6819
Epoch 04/5 | loss 1.6520 | meta_acc 0.7053 | meta_f1 0.7043
Epoch 05/5 | loss 1.4703 | meta_acc 0.6865 | meta_f1 0.6860
Epoch 01/15 | loss 1.6243 | meta_acc 0.7335 | meta_f1 0.7319
Epoch 02/15 | loss 1.5086 | meta_acc 0.7273 | meta_f1 0.7252
Epoch 03/15 | loss 1.3016 | meta_acc 0.7524 | meta_f1 0.7476
Epoch 04/15 | loss 1.3362 | meta_acc 0.7555 | meta_f1 0.7528
Epoch 05/15 | loss 1.3017 | meta_acc 0.7492 | meta_f1 0.7464
Epoch 06/15 | loss 1.2110 | meta_acc 0.7680 | meta_f1 0.7663
Epoch 07/15 | loss 1.1278 | meta_acc 0.7649 | meta_f1 0.7633
Epoch 08/15 | loss 1.0834 | meta_acc 0.7586 | meta_f1 0.7574
Epoch 09/15 | loss 1.0910 | meta_acc 0.7712 | meta_f1 0.7697
Epoch 10/15 | loss 1.0097 | meta_acc 0.7900 | meta_f1 0.7887
Epoch 11/15 | loss 1.0401 | meta_acc 0.7743 | meta_f1 0.7720
Epoch 12/15 | loss 1.0026 | m

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Epoch 01/5 | loss 1.1970 | meta_acc 0.6740 | meta_f1 0.6659
Epoch 02/5 | loss 0.9218 | meta_acc 0.7273 | meta_f1 0.7157
Epoch 03/5 | loss 0.8247 | meta_acc 0.7618 | meta_f1 0.7589
Epoch 04/5 | loss 0.7812 | meta_acc 0.7555 | meta_f1 0.7509
Epoch 05/5 | loss 0.7692 | meta_acc 0.7712 | meta_f1 0.7676
Epoch 01/15 | loss 0.7516 | meta_acc 0.7900 | meta_f1 0.7865
Epoch 02/15 | loss 0.6924 | meta_acc 0.7994 | meta_f1 0.7965
Epoch 03/15 | loss 0.6632 | meta_acc 0.8056 | meta_f1 0.8026
Epoch 04/15 | loss 0.6333 | meta_acc 0.8150 | meta_f1 0.8126
Epoch 05/15 | loss 0.6098 | meta_acc 0.8119 | meta_f1 0.8093
Epoch 06/15 | loss 0.5872 | meta_acc 0.8150 | meta_f1 0.8124
Epoch 07/15 | loss 0.5658 | meta_acc 0.8213 | meta_f1 0.8189
Epoch 08/15 | loss 0.5690 | meta_acc 0.8307 | meta_f1 0.8282
Epoch 09/15 | loss 0.5396 | meta_acc 0.8307 | meta_f1 0.8288
Epoch 10/15 | loss 0.5165 | meta_acc 0.8370 | meta_f1 0.8355
Epoch 11/15 | loss 0.5269 | meta_acc 0.8339 | meta_f1 0.8318
Epoch 12/15 | loss 0.5144 | m

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 01/5 | loss 0.7024 | meta_acc 0.9310 | meta_f1 0.9302
Epoch 02/5 | loss 0.3788 | meta_acc 0.9248 | meta_f1 0.9242
Epoch 03/5 | loss 0.3186 | meta_acc 0.9467 | meta_f1 0.9463
Epoch 04/5 | loss 0.2807 | meta_acc 0.9530 | meta_f1 0.9528
Epoch 05/5 | loss 0.2523 | meta_acc 0.9687 | meta_f1 0.9686
Epoch 01/15 | loss 0.2443 | meta_acc 0.9592 | meta_f1 0.9589
Epoch 02/15 | loss 0.2381 | meta_acc 0.9687 | meta_f1 0.9686
Epoch 03/15 | loss 0.2278 | meta_acc 0.9718 | meta_f1 0.9717
Epoch 04/15 | loss 0.2290 | meta_acc 0.9687 | meta_f1 0.9686
Epoch 05/15 | loss 0.2295 | meta_acc 0.9592 | meta_f1 0.9591
Epoch 06/15 | loss 0.2264 | meta_acc 0.9624 | meta_f1 0.9622
Epoch 07/15 | loss 0.2270 | meta_acc 0.9592 | meta_f1 0.9591
Epoch 08/15 | loss 0.2221 | meta_acc 0.9592 | meta_f1 0.9591
Early stopping.
[Swin-Tiny] META Acc: 0.9718 | META Macro-F1: 0.9717

Best single on META: ('Swin-Tiny', 0.9717868338557993, 0.9717484295806526)

Weighted soft-vote on META: 0.9561128526645768 0.9558504765539506 

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Stacking on META: 0.9686520376175548 0.9685872551361683

Chosen strategy (learned on META): Stacking(LogReg)

FINAL VALID RESULTS (NO LEAKAGE)
Strategy: Stacking(LogReg)
Valid Accuracy: 0.9750
Valid Macro-F1: 0.9748

Confusion Matrix:
 [[100   0   0   0]
 [  0 100   0   0]
 [  3   0  92   5]
 [  0   0   2  98]]

Classification Report:
               precision    recall  f1-score   support

         amd       0.97      1.00      0.99       100
    cataract       1.00      1.00      1.00       100
    diabetes       0.98      0.92      0.95       100
      normal       0.95      0.98      0.97       100

    accuracy                           0.97       400
   macro avg       0.98      0.97      0.97       400
weighted avg       0.98      0.97      0.97       400

